# 실습 5: 센서 590개 진단표 만들기
- 상황: 관리도로 한 개는 봤는데 나머지 589개가 남았다
- 목표: 열 하나를 한 줄로 요약해 쓸 수 없는 열을 걸러낸다

## Step 0. 폴더와 노트북 만들고 데이터 불러오기

In [1]:
import pandas as pd

df = pd.read_csv('../../data/04_secom.csv')

print('행 수, 열 수:', df.shape)

행 수, 열 수: (1567, 592)


## Step 1. 진단 항목 정하기

### 용어 풀이 - 열을 진단할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 진단표 | 열 하나가 한 줄이 되도록 요약한 표. 590개 열이 590줄이 된다 |
| 빈칸 비율 (결측률) | 그 열에서 값이 비어 있는 칸의 비율 |
| 값 종류 수 | 그 열에 서로 다른 값이 몇 가지 들어 있는지 |
| 상수열 | 값 종류가 1개뿐인 열. 처음부터 끝까지 같은 값만 나온다 |
| 스케일 | 값의 크기 단위. 어떤 열은 0~1, 어떤 열은 수천이라 그대로 비교하면 안 된다 |
| 임계값 | 버릴지 말지를 가르는 경계 숫자. 정답이 없어 사람이 정한다 |
| 1차 선별 | 쓸 수 없는 열을 먼저 떨어내는 단계. 쓸모를 따지는 건 그다음이다 |

[내 진단표에 넣을 항목]<br>
1. 빈칸 비율   2. 값 종류 수   3. 표준편차   4. 최소와 최대

---

## Step 2. 열 하나로 먼저 직접 계산해보기

### 열 하나로 진단 네 값 구해보기
590개에 시키기 전에, 한 열로 계산 방법을 익힌다.

In [2]:
# 앞 실습에서 골랐던 센서 이름을 다시 넣는다 (아무 센서나 괜찮습니다)
센서 = "sensor_089"

# isna() — 빈칸이면 참(True), 아니면 거짓(False)
# mean() — 참/거짓의 평균은 곧 참의 비율이 된다 (참=1, 거짓=0이라서)
빈칸비율 = df[센서].isna().mean() * 100

# nunique() — 서로 다른 값이 몇 종류인지 센다
값종류수 = df[센서].nunique()

# std() — 값이 평균에서 얼마나 흩어져 있는지
표준편차 = df[센서].std()

print("열 이름:", 센서)
print("빈칸 비율:", round(빈칸비율, 2), "%")
print("값 종류 수:", 값종류수)
print("표준편차:", round(표준편차, 4))
print("최소~최대:", df[센서].min(), "~", df[센서].max())

열 이름: sensor_089
빈칸 비율: 0.0 %
값 종류 수: 973
표준편차: 53.5373
최소~최대: 1627.4714 ~ 2105.1823


### 문법 노트 - 열 하나를 숫자로 줄이기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| .isna() | 빈칸이면 참, 아니면 거짓 | 빈칸을 세려면 먼저 참/거짓으로 바꿔야 한다 |
| .mean() | 평균 | 참/거짓에 쓰면 참의 비율이 된다 |
| .nunique() | 서로 다른 값의 개수 | 1이면 상수열. 이걸로 바로 판정된다 |
| .std() | 표준편차 | 0에 가까우면 거의 안 변하는 열 |
| .min() .max() | 가장 작은 값 / 가장 큰 값 | 값의 범위. 스케일 감을 잡는다 |

**.isna().mean()이 왜 비율이 되나?**<br>
참을 1, 거짓을 0으로 놓고 평균을 내기 때문이다.<br>
100개 중 4개가 빈칸이면 (1+1+1+1+0+0+...)/100 = 0.04 이므로 4%.<br>
개수를 세고 전체로 나누는 두 단계가 한 줄에 들어간 셈이다.<br>
앞으로 "비율을 구한다" 하면 거의 이 형태가 나온다.

---

## Step 3. 590개 전체는 AI에게 시키기

In [3]:
sensor_cols = [c for c in df.columns if c.startswith('sensor_')]

rows = []
for c in sensor_cols:
    s = df[c]
    rows.append({
        '열 이름': c,
        '빈칸 비율(%)': round(s.isna().mean() * 100, 2),
        '값 종류 수': s.nunique(),
        '표준편차': round(s.std(), 4),
        '최솟값': s.min(),
        '최댓값': s.max(),
    })

sensor_diagnosis = pd.DataFrame(rows)

print('sensor_diagnosis 전체 행 수:', len(sensor_diagnosis))

sensor_diagnosis.sort_values('빈칸 비율(%)', ascending=False).head(10)

sensor_diagnosis 전체 행 수: 590


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
292,sensor_293,91.19,92,0.0115,0.0041,0.0831
293,sensor_294,91.19,138,137.6925,82.3233,879.2260
158,sensor_159,91.19,138,406.8488,234.0996,2505.2998
157,sensor_158,91.19,128,0.0395,0.0118,0.2876
492,sensor_493,85.58,226,1.7593,4.8882,21.0443
85,sensor_086,85.58,97,0.0029,0.1053,0.1184
358,sensor_359,85.58,20,0.0004,0.0017,0.0047
220,sensor_221,85.58,69,0.0020,0.0057,0.0240
244,sensor_245,64.96,66,0.0846,0.0003,1.9844
517,sensor_518,64.96,543,4.8907,0.2880,113.2758


---

## Step 4. 위아래를 훑어보며 기준 세우기

In [4]:
낮은순_10 = sensor_diagnosis.sort_values('빈칸 비율(%)', ascending=True).head(10)

상수열_수 = (sensor_diagnosis['값 종류 수'] == 1).sum()
표준편차0_수 = (sensor_diagnosis['표준편차'] == 0).sum()

print('값 종류 수가 1개인 열(상수열) 개수:', 상수열_수)
print('표준편차가 0인 열 개수:', 표준편차0_수)
print('\n빈칸 비율 낮은 순 상위 10개:')

낮은순_10

값 종류 수가 1개인 열(상수열) 개수: 116
표준편차가 0인 열 개수: 116

빈칸 비율 낮은 순 상위 10개:


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
522,sensor_523,0.0,1562,7.1044,2.6811,137.9838
527,sensor_528,0.0,1549,1.8887,2.1700,14.4479
526,sensor_527,0.0,1514,0.9584,0.1705,8.2037
524,sensor_525,0.0,1543,20.6634,1.3104,818.0005
520,sensor_521,0.0,1536,5.7024,0.3121,111.7365
523,sensor_524,0.0,1040,4.1476,0.0258,111.3330
521,sensor_522,0.0,9,103.1230,0.0000,1000.0000
493,sensor_494,0.0,572,0.9739,0.8330,9.4024
392,sensor_393,0.0,109,0.0030,0.0005,0.0229
495,sensor_496,0.0,964,3.2600,1.7720,107.6926


[내가 정한 1차 선별 기준]<br>
1. 빈칸 비율이 [50]% 이상인 열은 버린다<br>
2. 값 종류가 1개인 열은 버린다 (상수열)<br>
3. 표준편차가 [0.001] 이하인 열은 버린다

---

## Step 5. 기준대로 걸러내기

In [5]:
조건1_빈칸 = sensor_diagnosis['빈칸 비율(%)'] >= 50
조건2_상수 = sensor_diagnosis['값 종류 수'] == 1
조건3_표준편차 = sensor_diagnosis['표준편차'] <= 0.001

print('조건1 (빈칸 비율 50% 이상):', 조건1_빈칸.sum(), '개')
print('조건2 (값 종류 수 1개):', 조건2_상수.sum(), '개')
print('조건3 (표준편차 0.001 이하):', 조건3_표준편차.sum(), '개')

# | (OR) — 세 조건 중 하나라도 걸리면 버릴 열. 겹치는 열은 한 번만 세어진다
버릴_행 = 조건1_빈칸 | 조건2_상수 | 조건3_표준편차
print('\n중복을 뺀 버릴 열 총 개수:', 버릴_행.sum(), '개')

# 원본 sensor_diagnosis는 그대로 두고, 걸러낸 결과를 새 이름으로 만든다
sensor_diagnosis_filtered = sensor_diagnosis[~버릴_행].reset_index(drop=True)

print('\n원본 sensor_diagnosis 행 수 (그대로):', len(sensor_diagnosis))
print('걸러낸 뒤 sensor_diagnosis_filtered 행 수:', len(sensor_diagnosis_filtered))

sensor_diagnosis_filtered.head(10)

조건1 (빈칸 비율 50% 이상): 28 개
조건2 (값 종류 수 1개): 116 개
조건3 (표준편차 0.001 이하): 127 개

중복을 뺀 버릴 열 총 개수: 154 개

원본 sensor_diagnosis 행 수 (그대로): 590
걸러낸 뒤 sensor_diagnosis_filtered 행 수: 436


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
0,sensor_001,0.38,1520,73.6218,2743.2400,3356.3500
1,sensor_002,0.45,1504,80.4077,2158.7500,2846.4400
2,sensor_003,0.89,507,29.5132,2060.6600,2315.2667
3,sensor_004,0.89,518,441.6916,0.0000,3715.0417
4,sensor_005,0.89,503,56.3555,0.6815,1114.5366
5,sensor_007,0.89,510,6.2372,82.1311,129.2522
6,sensor_008,0.57,89,0.0090,0.0000,0.1286
7,sensor_009,0.13,1208,0.0739,1.1910,1.6564
8,sensor_010,0.13,568,0.0151,-0.0534,0.0749
9,sensor_011,0.13,392,0.0093,-0.0349,0.0530


[1차 선별 결과]<br>
시작 : 센서 590개<br>
- 빈칸 과다로 제외 : [28]개<br>
- 상수열로 제외 : [116]개<br>
- 거의 안 변해서 제외 : [127]개<br>
(중복 제외) 총 제외 : [154]개<br>
남은 센서 : [436]개

---

## Step 6. 하나만 직접 확인하기

### 상수열 하나를 직접 확인
걸러낸 열이 정말 값이 하나뿐인지 눈으로 본다.

In [6]:
# value_counts() — 어떤 값이 각각 몇 번 나오는지 세어준다
# 따옴표 안은 예시입니다. AI가 알려준 상수열 목록에서 아무거나 하나로 바꾸세요
df["sensor_043"].value_counts()

sensor_043
70.0    1566
Name: count, dtype: int64

---

## Step 7. 아직 몇 개가 남았나

590개에서 \[436\]개로 줄었다. 그런데 이것도 관리도를 [436]개 그려야 한다.<br>
더 줄일 방법이 있을까? : [비슷하게 움직이는 센서끼리는 하나만 봐도 될 것 같다.<br>
그리고 판정과 아무 관계없는 열은 아예 뺄 수 있지 않을까]

---
## 직접 해보기 (도전) - 기준을 하나만 바꿔보면

- 상황: 빈칸 비율 기준을 내가 정했지만, 좁히면 얼마나 달라지는지는 아직 모른다
- 할 일: 기준을 하나만 좁혀서 다시 걸러내고 숫자를 비교한다
- 결과물: 세 줄짜리 비교표 1개

In [7]:
# 상수열 조건과 표준편차 조건은 두 기준 모두 동일하게 적용 (원본 sensor_diagnosis만 읽고, 수정하지 않는다)
상수열_조건 = sensor_diagnosis['값 종류 수'] == 1
표준편차_조건 = sensor_diagnosis['표준편차'] <= 0.001

def 기준별_요약(기준_이름, 빈칸_임계값):
    빈칸_조건 = sensor_diagnosis['빈칸 비율(%)'] >= 빈칸_임계값
    버릴_행 = 빈칸_조건 | 상수열_조건 | 표준편차_조건
    return {
        '기준': 기준_이름,
        '빈칸 과다로 제외': 빈칸_조건.sum(),
        '총 제외(중복 제거)': 버릴_행.sum(),
        '남는 센서 열': (~버릴_행).sum(),
    }

비교표 = pd.DataFrame([
    기준별_요약('A (50% 이상 제외)', 50),
    기준별_요약('B (30% 이상 제외)', 30),
])
비교표

,기준,빈칸 과다로 제외,총 제외(중복 제거),남는 센서 열
0,A (50% 이상 제외),28,154,436
1,B (30% 이상 제외),32,158,432


**인사이트**: 빈칸 비율 기준을 50%→30%로 좁혀도 총 제외 열이 154→158개(4개 차이)로 거의 안 늘어난다 — 30~50% 구간에 걸리는 센서가 몇 안 되므로, 1차 선별 결과는 빈칸 기준값 선택에 크게 민감하지 않다.

### 빈칸 기준 A vs B

| 항목 | A (50%) | B (30%) |
|---|---|---|
| 빈칸 과다로 제외 | [28] | [32] |
| 중복 뺀 총 제외 | [154] | [158] |
| 남는 센서 열 수 | [436] | [432] |